## Exploratory Data Analysis (EDA)

**Версия для самостоятельной работы**


В любой задаче, связанной с анализом данных, первым шагом всегда становится знакомство с самим датасетом. Прежде чем строить модели или делать выводы, важно оценить структуру, полноту и качество информации. Такой осмысленный подход к предварительному изучению данных получил название *Exploratory Data Analysis (EDA)*, он позволяет выявить проблемы, обнаружить закономерности и определить направления для дальнейшей работы.


### Цели EDA

**EDA (Exploratory Data Analysis — разведочный анализ данных)** помогает понять, с какими данными мы имеем дело, насколько они пригодны для анализа и в каком направлении двигаться дальше. В рамках этого анализа мы ставим следующие цели:

---

#### 1. Проверить качество и надёжность данных
- Обнаружить пропущенные, дубликатные или странные значения.
- Найти несоответствия в формате (например, `'male'` и `'Male'` в колонке пола).
- Определить, какие признаки не несут полезной информации или имеют подозрительное распределение.

---

#### 2. Оценить распределение признаков и пригодность данных к анализу
- Проверить, насколько сбалансированы классы или категории.
- Найти колонки с одним уникальным значением или нерепрезентативным содержанием.
- Убедиться, что данные позволяют ответить на поставленные вопросы.
  - *Пример*: если нужно сравнить интеллект котов и собак, а котов всего 3, — такой анализ не имеет смысла.
  - *Пример*: если нужно проверить, реже ли имеют детей образованные люди, а в выборке перекос по возрасту и доходу — нужно учитывать это в дальнейшем анализе или запросить другие данные.

  Пример из жизни: сейчас в Великобритании женщины 18-24 лет зарабатывают больше мужчин того же возраста на 10%. Если бы нам дали датасет, в котором мужчины и женщины имеют разное возрастное распределение, то анализ был бы не совсем честным.

---

#### 3. Сформулировать гипотезы, связанные с целевой переменной
- Предположить, какие признаки могут влиять на целевую переменную.
- Подготовиться к следующему этапу — построению модели или проверке статистических зависимостей.
- *Пример гипотезы*: «Образованные люди реже имеют детей», «Стаж влияет на уровень зарплаты», и т.п.


**📦 Задание 0: Импорт библиотек**

Запустите ячейку ниже для импорта необходимых библиотек.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import math
import scipy.stats as sps
%matplotlib inline

sns.set(palette="pastel", style='whitegrid', font_scale=1.3)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


### Данные retail

Аналитиков часто нанимают в **retail**, потому что маркетплейсы генерируют тонны данных — о товарах, действиях пользователей, продажах. Это отличная среда для аналитики, где много возможностей для экспериментов, тестов и оптимизаций.

Для бизнеса аналитик — это не просто "человек с графиками". Это специалист, который помогает **принимать решения**, опираясь на данные. Его задача — находить закономерности, проверять гипотезы и обеспечивать доказательную базу для бизнес-решений.

---

**Описание датасета**

Мы будем работать с [датасетом маркетинговой кампании](https://www.kaggle.com/datasets/ahsan81/superstore-marketing-campaign-dataset).

Ниже — описание признаков, данные собраны за последние два года до прошлогодней кампании, `Response` — отклик клиента на прошлую коммуникацию:

| Название столбца         | Описание                                                                 |
|---------------------------|--------------------------------------------------------------------------|
| **Response**              | Целевая переменная: `1`, если клиент принял предложение, иначе `0`       |
| **ID**                    | Уникальный идентификатор клиента                                        |
| **Year_Birth**            | Год рождения клиента                                                    |
| **Complain**              | `1`, если клиент жаловался за последние 2 года                           |
| **Dt_Customer**           | Дата регистрации клиента в компании                                     |
| **Education**             | Уровень образования клиента                                             |
| **Marital**               | Семейное положение клиента                                              |
| **Kidhome**               | Количество маленьких детей в семье                                      |
| **Teenhome**              | Количество подростков в семье                                           |
| **Income**                | Годовой доход домохозяйства клиента                                     |
| **MntFishProducts**       | Потрачено на рыбу за последние 2 года                                   |
| **MntMeatProducts**       | Потрачено на мясо                                                       |
| **MntFruits**             | Потрачено на фрукты                                                     |
| **MntSweetProducts**      | Потрачено на сладости                                                   |
| **MntWines**              | Потрачено на вино                                                       |
| **MntGoldProds**          | Потрачено на золотые товары                                             |
| **NumDealsPurchases**     | Количество покупок по скидкам                                           |
| **NumCatalogPurchases**   | Количество покупок по каталогу (доставка)                               |
| **NumStorePurchases**     | Количество покупок в офлайн-магазинах                                   |
| **NumWebPurchases**       | Количество покупок через сайт                                           |
| **NumWebVisitsMonth**     | Количество визитов на сайт за последний месяц                           |
| **Recency**               | Сколько дней назад была последняя покупка                               |

---

**Вопрос бизнеса**

Магазин хочет понять:

> **Насколько вероятно, что покупатель ответит положительно на рекламное предложение?**

Если удастся найти закономерности, можно **таргетировать кампанию** только на тех, кто с высокой вероятностью откликнется — и тем самым сократить издержки.

---

**Условия кампании**

Рекламная кампания: клиент получает **скидочную карту за 500 у.е.**, которая дает скидку на 20% от стоимости (в другие дни цена такой карты — 900).

---

**Гипотезы**

&#x2753; **Вопрос** &#x2753;

>Какие признаки, по вашему мнению, могут влиять на то, откликнется ли клиент?

>- Чем больше клиент тратит, тем выше вероятность, что он купит карту?
- Клиенты с детьми экономнее?
- Пожилые клиенты менее склонны участвовать в акциях?
>- Жалобы в прошлом снижают вероятность отклика? Или наоборот?

---

Начнем с исследования данных!


### 1. Работа с датасетом

Теперь посмотрим на данные: как они выглядят, что с ними можно сделать, чтобы были более удобные.


**📝 Задание 1.1: Загрузка данных**

Загрузите данные из файла `superstore_data.csv` и выведите первые строки датафрейма.

💡 *Подсказка: используйте `pd.read_csv()` для загрузки*


In [ ]:
# Твой код здесь
data = pd.read_csv("C:\Users\Lenovo\Downloads\Telegram Desktop\superstore_data.csv")
data


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (ipython-input-3754188273.py, line 2)

**📝 Задание 1.2: Информация о данных**

Выведите информацию о типах данных и количестве непустых значений в каждом столбце.

💡 *Подсказка: метод `.info()` покажет типы и пропуски*


In [ ]:
# Твой код здесь



У нас есть несколько пропущенных значений в столбце дохода. Что можно сделать с пропусками?

<details>

  - выкинуть строки с пропусками

  - заполнить абсурдными значениями (-1 или 1e10, например)  

  - заполнить медианой, средним или каким-то другим способом  

Это решение будет зависеть от поставленной задачи.
</details>

Так как пропусков совсем немного, можно их просто исключить.


**📝 Задание 1.3: Удаление пропусков**

Удалите строки, где значение `Income` пропущено (NaN).

💡 *Подсказка: используйте `.notna()` для фильтрации или `.dropna()` с параметром `subset`*


In [ ]:
# Твой код здесь
data = ...


**📝 Задание 1.4: Проверка дубликатов**

Проверьте, есть ли в данных:
1. Полные дубликаты строк
2. Дубликаты по столбцу `Id` (один и тот же клиент несколько раз)

💡 *Подсказка: используйте метод `.duplicated()` и `.sum()` для подсчёта*


In [ ]:
# Проверка полных дубликатов
# Твой код здесь


# Проверка дубликатов по Id
# Твой код здесь



Видим, что в данных присутствует личная информация о клиентах — возраст, семейное положение, образование, а также поведенческие характеристики: количество и сумма покупок, покупки со скидками, покупки в магазине и на сайте, количество просмотров и т.д.

&#x2753; **Вопрос** &#x2753;

> Какие новые признаки можно добавить для улучшения анализа данных?

<details> <summary>Открыть подсказку</summary>

Примеры новых признаков:

* Объединить количество детей и подростков в общий признак `Total_Children`
* Рассчитать `Total_Spent` — общую сумму покупок, просуммировав траты по всем категориям
* Преобразовать дату регистрации в datetime формат

</details>


**📝 Задание 1.5: Создание новых признаков**

Создайте следующие новые признаки:
1. `Dt_Customer` — преобразуйте в datetime (формат `'%m/%d/%Y'`)
2. `Total_Children` — сумма `Kidhome` и `Teenhome`
3. `Total_Spent` — сумма всех трат (`MntWines`, `MntFruits`, `MntMeatProducts`, `MntFishProducts`, `MntSweetProducts`, `MntGoldProds`)

💡 *Подсказка: для суммы нескольких столбцов используйте `data[список_столбцов].sum(axis=1)`*


In [ ]:
# Твой код здесь
data['Dt_Customer'] = ...
data['Total_Children'] = ...
data['Total_Spent'] = ...


У нас есть категориальные и числовые признаки.

**Категориальные** — это признаки, которые принимают ограниченное число значений (например: пол, образование, "да/нет"), и обычно не имеют арифметического смысла — их не стоит складывать, делить или напрямую сравнивать по величине.

**Числовые признаки**, наоборот, имеют количественную природу — их можно усреднять, сравнивать (больше/меньше), использовать в вычислениях.

Разделим сразу признаки на группы:


In [ ]:
# Категориальные и числовые признаки (уже определены для вас)
cat_columns = ['Education', 'Marital_Status', 'Kidhome', 'Teenhome', 'Total_Children', 'Response', 'Complain']
num_columns = ['Year_Birth', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds',
               'NumDealsPurchases', 'NumCatalogPurchases', 'NumStorePurchases',
               'NumWebVisitsMonth', 'Total_Spent', 'Income', 'Recency']


Обратим внимание, что число детей мы отнесли к категориальному признаку, хотя в колонке числа 0, 1, 2, что формально является числами. А год рождения (тоже дискретный) — к числовым.

&#x2753; **Вопрос** &#x2753;

> Почему так?

<details> <summary>Открыть объяснение</summary>

Число детей — это ограниченное и неравномерное множество значений (например, 0, 1, 2), где между соседними числами нет очевидной линейной зависимости.

Эти значения часто используют как категории — например, можно закодировать через one-hot.

Возраст, наоборот, — количественный признак: его можно усреднять, вычитать, сравнивать, он ближе к непрерывной шкале.
</details>


### 2. Работа с категориальными данными

Начнем с двух визуализаций. Возьмем колонку `Education` и визуализируем с помощью `Bar Plot` и `Pie Chart`:

**Bar plot** — Классическая гистограмма для категориальных признаков. Показывает, сколько раз встречается каждое значение. Удобно использовать для первичного анализа распределений.

**Pie Chart** — Круговая диаграмма для визуализации долей каждой категории от общего числа. Полезна, когда нужно показать, какую часть занимает категория в общей структуре, но плохо читается при большом количестве категорий.

⚠️ *С круговыми диаграммами надо быть осторожнее: человеческий глаз не может достоверно сравнить размеры сегментов. Это может привести к ложным выводам.*


**📝 Задание 2.1: Bar Plot для Education**

Постройте барплот для столбца `Education`, показывающий процент от общего числа.

💡 *Подсказки:*
- *Используйте `value_counts(normalize=True) * 100` для получения процентов*
- *`sns.barplot(x=..., y=..., palette='pastel')` для построения графика*


In [1]:
data.Education

NameError: name 'data' is not defined

In [ ]:
# Твой код здесь
edu_counts =
sns.barplot(...)
plt.ylabel('% от общего числа')
plt.title('Распределение Education')
plt.xticks(rotation=45)
plt.show()


**📝 Задание 2.2: Pie Chart для Education**

Постройте круговую диаграмму для столбца `Education`.

💡 *Подсказки:*
- *Используйте `plt.pie()` с параметрами `labels`, `autopct='%1.1f%%'`*
- *`plt.axis('equal')` делает круг ровным*


In [ ]:
# Твой код здесь
edu_counts = data['Education'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(
    ...,  # данные
    labels=...,  # подписи
    autopct='%1.1f%%',
    startangle=90,
    colors=sns.color_palette('pastel')
)
plt.title('Распределение Education')
plt.axis('equal')
plt.show()


**📝 Задание 2.3: Барплоты для всех категориальных признаков**

Постройте барплоты для всех категориальных признаков из списка `cat_columns`. Добавьте аннотации с количеством и процентом.

💡 *Подсказки:*
- *Используйте `sns.countplot(data=data, x=col, ax=ax, order=order)` для построения*
- *`ax.annotate()` для добавления текста над столбиком*
- *`ax.patches` содержит все столбики графика*


In [ ]:
# Твой код здесь
n_cat = len(cat_columns)
cols = 3
rows = math.ceil(n_cat / cols)

fig_cat, axes_cat = plt.subplots(rows, cols, figsize=(cols * 10, rows * 7))
axes_cat = axes_cat.flatten()

for i, col in enumerate(cat_columns):
    ax = axes_cat[i]
    order = data[col].value_counts().index  # Порядок по убыванию
    counts = data[col].value_counts()
    total = len(data)

    # Построй countplot здесь
    # sns.countplot(...)

    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)

    # Добавь аннотации с количеством и процентом
    for p in ax.patches:
        count = int(p.get_height())
        percent = 100 * count / total
        # ax.annotate(...)
        pass

# Удаляем лишние оси
for j in range(i + 1, len(axes_cat)):
    fig_cat.delaxes(axes_cat[j])

fig_cat.suptitle("Категориальные признаки - барплоты с количеством и процентами", fontsize=24)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


&#x2753; **Вопрос** &#x2753;

> Что можно сказать по этим графикам?

<details>
<summary> Открыть объяснение </summary>

* Большинство имеют высшее образование, магистров и кандидатов больше трети.
* Больше половины состоят в отношениях
* Половина семей имеет одного ребенка, меньше трети — без детей
* На предыдущую кампанию откликнулось 15%, что неплохой показатель для рекламной кампании
* Менее одного процента жаловались на магазин, значит покупатели в целом всем довольны
</details>

**Что можно изменить в этих данных?**
* Категории `Single` и `Alone` — взаимозаменяемые, можно объединить
* `YOLO` и `Absurd` можно исключить, так как непонятно, что это означает, а их количество очень маленькое


**📝 Задание 2.4: Очистка категориальных данных**

1. Объедините категории `Alone` и `Single` в одну (`Single`)
2. Удалите строки с `YOLO` и `Absurd` в столбце `Marital_Status`

💡 *Подсказки:*
- *`.replace({'старое': 'новое'})` для замены значений*
- *`~data['col'].isin([список])` для фильтрации (исключения) значений*


In [ ]:
# Твой код здесь
# Объединяем категории
data['Marital_Status'] = ...

# Удаляем нерелевантные значения
data = ...


**Дополнительные способы визуализации категориальных данных:**

**Stacked bar plot** — Сложенные столбцы позволяют одновременно анализировать распределение одного признака внутри другого. Например, как семейное положение распределено по уровням образования.

**Crosstab heatmap** — Тепловая карта по кросстаблице двух категориальных признаков. Показывает частоты или доли в виде цветовой интенсивности.


**📝 Задание 2.5: Анализ зависимости Education и Marital_Status**

Постройте:
1. Stacked bar plot — семейное положение в зависимости от образования
2. Heatmap — кросстаблицу Education vs Marital_Status

💡 *Подсказки:*
- *`pd.crosstab(index, columns)` создаёт кросстаблицу*
- *`normalize='index'` нормализует по строкам (в процентах)*
- *`.plot(kind='bar', stacked=True)` для stacked bar*
- *`sns.heatmap(data, annot=True, fmt='d')` для тепловой карты*


In [ ]:
# Stacked Bar
# Создаем кросс-таблицу с нормализацией по строкам (в процентах)
cross_tab = pd.crosstab(..., ..., normalize='index') * 100

# Твой код здесь: построй stacked bar plot
cross_tab.plot(kind='bar', stacked=True)
plt.ylabel('% внутри Education')
plt.title('Семейное положение в зависимости от образования')
plt.legend(title='Marital Status', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Heatmap
# Создаем кросс-таблицу с абсолютными значениями
cross_counts = pd.crosstab(..., ...)

# Твой код здесь: построй heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(..., annot=True, fmt='d')
plt.title('Кросстаблица: Education vs Marital_Status')
plt.ylabel('Education')
plt.xlabel('Marital Status')
plt.tight_layout()
plt.show()


### 3. Работа с числовыми данными

Возьмем признак `Income` и визуализируем несколькими способами.

---

**Методы визуализации распределения**

**Гистограмма** (`histplot`) — показывает, как часто встречаются значения в разных диапазонах. Полезна для понимания формы распределения.

**Box Plot (ящик с усами)** — показывает медиану, квартили 1 и 3 (границы ящика), выбросы (значения за пределами 1.5×IQR). Полезен для поиска аномалий и асимметрии.

**Violin Plot** — комбинирует KDE (гладкую плотность) и Boxplot (медиану, IQR). Показывает распределение и форму в одном графике.


**📝 Задание 3.1: Визуализация Income**

Постройте три графика для столбца `Income`:
1. Гистограмму (histplot)
2. Box plot
3. Violin plot

💡 *Подсказки:*
- *`sns.histplot(data['Income'], bins=30)` для гистограммы*
- *`sns.boxplot(y=data['Income'], width=0.2)` для боксплота*
- *`sns.violinplot(y=data['Income'], inner='box', width=0.2)` для violin*


In [ ]:
# Гистограмма
plt.figure(figsize=(8, 5))
# Твой код здесь
plt.title('Гистограмма дохода')
plt.xlabel('Доход')
plt.ylabel('Частота')
plt.show()

# Боксплот
plt.figure(figsize=(6, 4))
# Твой код здесь
plt.title('Boxplot дохода')
plt.ylabel('Доход')
plt.show()

# Violin plot
plt.figure(figsize=(6, 4))
# Твой код здесь
plt.title('Violin plot дохода')
plt.ylabel('Доход')
plt.show()


**Ядерная оценка плотности (KDE)**

Плавная оценка распределения, без резких столбиков. Хорошо показывает форму, пики и хвосты, но может быть неустойчива к выбросам.

**Смысл:** в каждую точку выборки поставили отмасштабированное ядро так, будто эта точка — центр ядра, а затем усреднили значения соседних точек с весами, заданными этим ядром. Вместо тысячи слов — [интерактивная иллюстрация](https://mathisonian.github.io/kde/).

Ядерные оценки плотности — KDE, Kernel Density Estimates — способ что-то понять о распределении, когда неизвестно ничего. Такого рода методы называют **непараметрическими**.


**📝 Задание 3.2: Гистограмма с KDE и теоретическим распределением**

Постройте гистограмму с плотностью для `Income`, добавьте KDE и кривую нормального распределения.

💡 *Подсказки:*
- *`sns.histplot(..., stat='density', kde=True)` для гистограммы с KDE*
- *`sns.rugplot()` добавляет штрихи снизу*
- *`sps.norm.fit(data)` вернёт (mu, sigma) для нормального распределения*
- *`sps.norm.pdf(x, mu, sigma)` даст значения плотности*


In [ ]:
plt.figure(figsize=(8, 5))

# Твой код здесь: гистограмма с плотностью и KDE
# sns.histplot(...)

# Rugplot (штрихи снизу)
# sns.rugplot(...)

# Теоретическая плотность по нормальному распределению
xmin, xmax = data['Income'].min(), data['Income'].max()
x = np.linspace(xmin, xmax, 500)
# params = sps.norm.fit(...)  # fit вернет (mu, sigma)
# pdf_fitted = sps.norm.pdf(x, *params)
# plt.plot(x, pdf_fitted, 'r--', label='Нормальное распределение (fit)')

plt.title('Оценка распределения дохода')
plt.xlabel('Доход')
plt.ylabel('Плотность')
plt.legend()
plt.show()


**📝 Задание 3.3: Гистограммы для всех числовых признаков**

Постройте гистограммы с KDE для всех признаков из `num_columns`.

💡 *Подсказка: используй цикл по признакам и `plt.subplots()` для сетки графиков*


In [ ]:
# Твой код здесь
n_num = len(num_columns)
cols = 3
rows = math.ceil(n_num / cols)

fig_hist, axes_hist = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
axes_hist = axes_hist.flatten()

for i, col in enumerate(num_columns):
    ax = axes_hist[i]
    # Построй гистограмму с KDE здесь
    # sns.histplot(...)
    pass

# Удаляем лишние оси
for j in range(i + 1, len(axes_hist)):
    fig_hist.delaxes(axes_hist[j])

fig_hist.suptitle("Числовые признаки - Гистограммы с KDE", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


**📝 Задание 3.4: Box Plots для всех числовых признаков**

Постройте боксплоты для всех признаков из `num_columns`, чтобы увидеть выбросы.


In [ ]:
# Твой код здесь
n_num = len(num_columns)
cols = 3
rows = math.ceil(n_num / cols)

fig_num, axes_num = plt.subplots(rows, cols, figsize=(cols * 5, rows * 2))
axes_num = axes_num.flatten()

for i, col in enumerate(num_columns):
    ax = axes_num[i]
    # Построй боксплот здесь
    # sns.boxplot(...)
    pass

# Удаляем пустые оси
for j in range(i + 1, len(axes_num)):
    fig_num.delaxes(axes_num[j])

fig_num.suptitle("Количественные признаки - Box Plots")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


&#x2753; **Вопрос** &#x2753;

> Мы видим, что в данных много выбросов. Надо ли их убирать? В каких признаках?

<details>
<summary> Кликни для показа ответа </summary>

Выбросы — это значения, которые значительно отличаются от основной массы данных. Их видно на боксплотах как точки, лежащие вне "усов".

**Много выбросов:**
* **Income** — очень сильный выброс (>600,000)
* **Total_Spent** — заметный выброс справа
* **MntWines, MntMeatProducts, MntGoldProds** — длинные хвосты

**Убирать или не убирать?**

Ответ всегда зависит от задачи. С одной стороны, выбросы могут исказить тренды, среднее, медианы, корреляции. С другой — могут быть важными контрибьюторами.

Если мы применяем методы, чувствительные к таким выбросам, то их надо либо удалить, либо выбрать устойчивые способы анализа.
</details>


### 4. Взаимодействия признаков друг с другом

Также можно рассмотреть распределения данных по двум осям-признакам с разделением по какому-то классу.

**PairGrid** — мощный инструмент для визуализации попарных отношений между переменными. Он создаёт сетку графиков, где каждый элемент показывает взаимосвязь между двумя переменными.

| Метод | Описание |
|-------|----------|
| `.map(func)` | Применяет функцию ко всем графикам |
| `.map_diag(func)` | Только диагональные графики |
| `.map_offdiag(func)` | Все графики кроме диагональных |
| `.map_lower(func)` | Графики ниже диагонали |
| `.map_upper(func)` | Графики выше диагонали |


**📝 Задание 4.1: PairGrid для числовых признаков**

Создайте PairGrid для числовых признаков с разделением по `Response`:
- На нижней диагонали — KDE (`sns.kdeplot`)
- На верхней диагонали — точечный график (`plt.scatter`)
- На диагонали — KDE распределения

💡 *Подсказка: используй `sns.PairGrid(data, hue='Response')` и методы `.map_lower()`, `.map_upper()`, `.map_diag()`*


In [ ]:
# Подготовим данные
subset_cols = num_columns + ['Response']
clean_data = data[subset_cols]

# Твой код здесь
# g = sns.PairGrid(clean_data, diag_sharey=False, hue='Response')
# g.map_lower(...)
# g.map_upper(...)
# g.map_diag(...)


**Немного про корреляционный анализ**

**Коэффициент корреляции** — это число от -1 до 1, которое показывает, насколько сильно две переменные зависят друг от друга **линейно**.

* **Близко к 1**: Чем больше одна переменная, тем больше другая
* **Близко к -1**: Чем больше одна переменная, тем меньше другая
* **Близко к 0**: Линейной связи нет

**Важно:** этот коэффициент "видит" только простые линейные зависимости. Сложные связи он может не заметить.


**📝 Задание 4.2: Корреляционная матрица (Heatmap)**

Постройте тепловую карту корреляций между числовыми признаками. Отфильтруйте слабые корреляции (|r| < 0.2) и покажите только нижний треугольник матрицы.

💡 *Подсказки:*
- *`data.corr(numeric_only=True)` вычисляет корреляции*
- *`np.triu(np.ones_like(corr, dtype=bool))` создаёт маску для верхнего треугольника*
- *`sns.heatmap(..., mask=mask, annot=True, fmt=".2f", cmap='coolwarm')` для тепловой карты*


In [ ]:
# Твой код здесь
# Вычисляем корреляции
corr = ...

# Убираем корреляции ближе к 0 (|r| < 0.2)
mask = np.abs(corr) < 0.2
filtered_corr = corr.mask(mask)

plt.figure(figsize=(14, 10))
# Маска для верхнего треугольника
mask_upper = np.triu(np.ones_like(filtered_corr, dtype=bool))

# Построй heatmap здесь
# sns.heatmap(...)

plt.title('Отфильтрованная корреляционная матрица (|r| > 0.2)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


&#x2753; **Вопрос** &#x2753;

> Какие интересные зависимости вы видите на корреляционной матрице?

<details>
<summary>Открыть ответ</summary>

- Доход значительно коррелирует с тратами, в том числе по категориям
- Чем больше у людей маленьких детей, тем меньше люди тратят в магазине
- С `Response` коррелируют `MntWines`, `MntMeatProducts`, `NumCatalogPurchases`
- Чем больше доход, тем меньше количество детей
- `NumDealsPurchases` — покупки по скидкам — коррелируют только с количеством детей. Видимо семьи с детьми более экономные
</details>


### 5. Таргетное рассмотрение

Можно таргетно рассмотреть зависимости признак-признак. `sns.jointplot` — отличный способ визуализировать связь двух числовых признаков.


**📝 Задание 5.1: JointPlot для Income и MntWines**

Постройте jointplot для зависимости `Income` и `MntWines`:
1. Сначала с `kind='kde'` (контурный график плотности)
2. Затем с точками (по умолчанию)

💡 *Подсказка: `sns.jointplot(x=..., y=..., kind='kde')` или просто `sns.jointplot(x=..., y=...)`*


In [ ]:
# Твой код здесь - KDE jointplot
# sns.jointplot(...)


In [ ]:
# Твой код здесь - scatter jointplot
# sns.jointplot(...)


**📝 Задание 5.2: Двумерное KDE с разделением по Response**

Создайте признак `Age` (возраст = 2020 - год рождения) и постройте двумерный KDE график Age vs Total_Spent с разделением по Response.

💡 *Подсказки:*
- *Создай `data['Age'] = 2020 - data['Year_Birth']`*
- *Используй два вызова `sns.kdeplot()` с разными фильтрами данных*
- *`cmap='Blues'` и `cmap='Reds'` для разных цветов*


In [ ]:
# Создаем признак Age
data['Age'] = ...

plt.figure(figsize=(12, 8))

# Твой код здесь: KDE для клиентов, которые не ответили (Response == 0)
# ax = sns.kdeplot(x=..., y=..., label="No Response", cmap='Blues', fill=False)

# Твой код здесь: KDE для клиентов, которые ответили (Response == 1)
# ax = sns.kdeplot(x=..., y=..., label="Responded", cmap='Reds', fill=False)

plt.title("Density Plot: Age vs Total Spending by Response")
plt.xlabel("Age")
plt.ylabel("Total Spending")
plt.ylim(0, 2500)
plt.legend()
plt.show()


**Боксплоты для сравнения по категориям**

Боксплоты полезны, чтобы сравнить числовой признак по категориям. Несколько таких ящиков можно нарисовать бок о бок для визуального сравнения.


**📝 Задание 5.3: Боксплоты Total_Spent по категориям**

Постройте боксплоты для `Total_Spent` в разрезе:
1. Education
2. Marital_Status

💡 *Подсказка: `sns.boxplot(x='категория', y='Total_Spent', data=data)`*


In [ ]:
# Твой код здесь - боксплот по Education
# sns.boxplot(...)
plt.title('Total Spent vs Education')
plt.xticks(rotation=45)
plt.show()

# Твой код здесь - боксплот по Marital_Status
# sns.boxplot(...)
plt.title('Total Spent vs Marital Status')
plt.xticks(rotation=45)
plt.show()


**📝 Задание 5.4: Боксплот с группировкой (hue)**

Постройте боксплот `Total_Spent` по `Total_Children` с группировкой по `Education`.

💡 *Подсказка: добавь параметр `hue='Education'` в `sns.boxplot()`*


In [ ]:
# Твой код здесь
plt.figure(figsize=(20, 10))
# sns.boxplot(x='Total_Children', y='Total_Spent', data=data, hue=...)
plt.title('Траты vs Total Children vs Education')
plt.xticks(rotation=45)
plt.show()


&#x2753; **Вопрос** &#x2753;

> Что можно сказать о зависимости трат от количества детей?

<details>
<summary>Открыть ответ</summary>

Люди, у которых есть дети тратят **значительно** меньше, причем тренд сохраняется с ростом числа детей и общий для всех ступеней образования.
</details>


### 6. Вспомним, зачем мы это делаем

Теперь посмотрим на нашу целевую переменную. Вспоминаем, что у магазина была задача оценить, можно ли таргетно запустить рекламную кампанию и понять, от чего зависит, откликнется ли клиент.

Напоминание: данные мы получили ***до*** прошлогодней кампании.


**📝 Задание 6.1: Визуализация Total_Spent по Response**

Постройте 4 графика для `Total_Spent` с разделением по `Response`:
1. KDE Plot (две кривые плотности)
2. Boxplot
3. Violin plot
4. Bar Plot (средние значения)

💡 *Подсказки:*
- *Для KDE: фильтруй данные по Response и строй две кривые*
- *Для boxplot/violin: используй `x='Response', y='Total_Spent'`*
- *Для bar: `data.groupby('Response')['Total_Spent'].mean()`*


In [ ]:
plt.figure(figsize=(18, 12))
colors = sns.color_palette('pastel')

# KDE Plot
plt.subplot(2, 2, 1)
# Твой код здесь: две кривые KDE для Response=1 и Response=0
# sns.kdeplot(data=..., label='Responded', fill=True, alpha=0.5)
# sns.kdeplot(data=..., label='Did Not Respond', fill=True, alpha=0.5)
plt.title('Density Plot: Total Spending')
plt.xlabel('Total Spending')
plt.ylabel('Density')
plt.legend()
plt.grid(True, alpha=0.3)

# Boxplot
plt.subplot(2, 2, 2)
# Твой код здесь
# sns.boxplot(x=..., y=..., data=data, width=0.5)
plt.title('Boxplot: Total Spending by Response')
plt.xticks([0, 1], ['Did Not Respond', 'Responded'])
plt.ylabel('Total Spending')
plt.grid(True, alpha=0.3)

# Violin plot
plt.subplot(2, 2, 3)
# Твой код здесь
# sns.violinplot(x=..., y=..., data=data, inner='quartile')
plt.title('Violin Plot: Total Spending by Response')
plt.xticks([0, 1], ['Did Not Respond', 'Responded'])
plt.ylabel('Total Spending')
plt.grid(True, alpha=0.3)

# Bar Plot (средние)
plt.subplot(2, 2, 4)
# avg_spent = data.groupby(...)[...].mean()
# sns.barplot(x=avg_spent.index, y=avg_spent.values, alpha=0.7)
plt.xticks([0, 1], ['Did Not Respond', 'Responded'])
plt.title('Average Spending by Response Group')
plt.ylabel('Average Total Spent')
plt.grid(True, alpha=0.3)

plt.tight_layout(pad=3.0)
plt.show()


**📝 Задание 6.2: Категориальные признаки с разделением по Response**

Постройте барплоты для всех категориальных признаков из `cat_columns` с разделением по `Response` (используй параметр `hue='Response'`).


In [ ]:
# Твой код здесь
n_cat = len(cat_columns)
cols = 3
rows = math.ceil(n_cat / cols)

fig_cat, axes_cat = plt.subplots(rows, cols, figsize=(cols * 10, rows * 7))
axes_cat = axes_cat.flatten()

for i, col in enumerate(cat_columns):
    ax = axes_cat[i]
    order = data[col].value_counts().index

    # Построй countplot с hue='Response' здесь
    # sns.countplot(data=data, x=col, ax=ax, order=order, hue=...)

    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)

for j in range(i + 1, len(axes_cat)):
    fig_cat.delaxes(axes_cat[j])

fig_cat.suptitle("Категориальные признаки с разделением по Response", fontsize=24)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


**📝 Задание 6.3: KDE числовых признаков с разделением по Response**

Постройте KDE графики для всех числовых признаков из `num_columns` с разделением по `Response`.

💡 *Подсказка: `sns.kdeplot(data=data, x=col, hue='Response', fill=True, alpha=0.5, common_norm=False)`*


In [ ]:
# Твой код здесь
n_num = len(num_columns)
cols = 3
rows = math.ceil(n_num / cols)

fig_kde, axes_kde = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
axes_kde = axes_kde.flatten()

for i, col in enumerate(num_columns):
    ax = axes_kde[i]
    # Построй kdeplot с hue='Response' здесь
    # sns.kdeplot(data=..., x=..., hue=..., fill=True, alpha=0.5, ax=ax, common_norm=False)
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.grid(True, alpha=0.3)

for j in range(i + 1, len(axes_kde)):
    fig_kde.delaxes(axes_kde[j])

fig_kde.suptitle("Количественные признаки - KDE в зависимости от отклика", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


In [ ]:
corr = ...

# Убираем корреляции ближе к 0
mask = np.abs(corr) < 0.05
filtered_corr = corr.mask(mask)

plt.figure(figsize=(min(2 + len(filtered_corr.columns), 16), 10))
mask_upper = np.triu(np.ones_like(filtered_corr, dtype=bool))

# всё, что мало коррелирует — прозрачное, + верх треуг
sns.heatmap(filtered_corr, mask=mask_upper, annot=True, fmt=".2f",
            cmap='coolwarm', center=0, linewidths=0.5, cbar_kws={"shrink": 0.8},
            annot_kws={"size": 8})

plt.title('Отфильтрованная корреляционная матрица', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

&#x2753; **Вопрос** &#x2753;

> Какие признаки отличаются у тех, кто откликнулся на кампанию, от тех, кто не откликнулся?

<details>
<summary>Открыть ответ</summary>

- Те, кто ответил покупают больше вина, мяса, фруктов и тд. Но эти параметры — корреляты дохода. Поэтому нужно учесть корреляцию и сделать акцент на доходе.
- Признак `Recency` сильно отличается между группами, хотя он ни с кем не коррелирует сильно. Те, кто "чаще" заходят в магазин будут с большей вероятностью отвечать на рекламу.
- Клиенты без детей чаще откликаются на предложения
</details>
